In [148]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, FunctionTransformer, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [149]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="'penalty' was deprecated in version 1.8",
    category=FutureWarning
)

## cross validation amb train

In [150]:
train = pd.read_csv('./train.csv')

In [151]:
X = train.drop(columns=["id","label"])
y = train["label"]

In [152]:
binary_columns = [
    "diabetes",
    "hypertension",
    "chronic_kidney_disease",
    "readmitted_30d"
]

int_columns = ["age", "systolic_bp", "diastolic_bp", "heart_rate", "glucose_mg_dl", "sodium_mmol_l", "respiratory_rate", "spo2"]

In [153]:
def feature_engineering(data):
    data = data.copy()

    #feature engineering without dropping
    data["mean_arterial_pressure"] = (
        data["systolic_bp"] + 2 * data["diastolic_bp"]
    ) / 3

    data["bmi_age_interaction"] = (
        data["bmi"] * data["age"]
    )

    data["pulse_pressure"] = data["systolic_bp"] - data["diastolic_bp"]

    data["log_length"] = np.log1p(data["length_of_stay_days"])

    data["log_creatinine"] = np.log(data["creatinine_mg_dl"])

    # yes no columns to 0 and 1
    for column in binary_columns:
        data[column] = data[column].map({"No": 0, "Yes": 1})

    # convert int and NaN to string
    def to_str(x):
        return x.astype("Int64").astype(str).where(x.notna(), "nan")

    #dictionary for new features
    new = {}

    # last digit of integer columns ("digit preference" of each LLM)
    for c in int_columns:
            new[c + "_ld"] = to_str(data[c] % 10)


    # tenths digit of 1-decimal columns, hundredths of creatinine, respiratory rate parity
    for c in ["bmi", "temperature_c", "hemoglobin_g_dl", "length_of_stay_days"]:
            new[c + "_ld"] = to_str((data[c] * 10).round() % 10)
    new["creat_dec2"] = to_str((data["creatinine_mg_dl"] * 100).round() % 10)
    new["rr_parity"] = to_str(data["respiratory_rate"] % 2)

    # exact value as a category (each LLM has its own "vocabulary" of values)
    for c in ["temperature_c", "bmi", "hemoglobin_g_dl", "length_of_stay_days",
            "creatinine_mg_dl"]:
        new[c + "_ex"] = data[c].round(2).astype(str).where(data[c].notna(), "nan")

    for c in ["num_medications", "prior_visits_12m"] + int_columns:
        new[c + "_ex"] = to_str(data[c])


    return pd.concat([data, pd.DataFrame(new, index=data.index)], axis=1)

In [154]:
X = feature_engineering(X)

In [155]:
# categorical and numerical features. no les llistem pq ara hi ha moltes

cat_cols = [c for c in X if X[c].dtype == object or str(X[c].dtype) in ("str", "string")]
num_cols = [c for c in X if c not in cat_cols]

In [156]:
#preprocesor and pipelines

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),

    # rare levels are grouped together -> keeps penalty=None from overfitting
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=10, sparse_output=False), cat_cols),
])

In [157]:
#model
model = Pipeline([("pre", preprocessor),
                  ("clf", LogisticRegression(penalty=None, max_iter=10000))])

In [158]:
#cross validation
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)
scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy for each fold:", scores.round(4))
print("Mean validation accuracy:", round(scores.mean(), 4))
print("Variation between folds:", round(scores.std(), 4))

Accuracy for each fold: [0.9458 0.9167 0.9333 0.9208 0.9    0.9104 0.9312 0.9188 0.9208 0.8979
 0.9146 0.9125 0.9229 0.9104 0.9188 0.9229 0.925  0.9125 0.9208 0.9104
 0.9021 0.9208 0.9333 0.9292 0.925 ]
Mean validation accuracy: 0.9191
Variation between folds: 0.0108


## to submit

In [160]:
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

In [161]:
X_train = train.drop(columns=["label","id"])
y_train = train["label"]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

apliquem la funció definida adalt

In [162]:
X_train = feature_engineering(X_train)
X_test = feature_engineering(X_test)

In [163]:
cat_cols = [c for c in X_train if X_train[c].dtype == object or str(X_train[c].dtype) in ("str", "string")]
num_cols = [c for c in X_train if c not in cat_cols]

In [164]:
#preprocesor and pipelines

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),

    # rare levels are grouped together -> keeps penalty=None from overfitting
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=10, sparse_output=False), cat_cols),
])

In [165]:
model = Pipeline([("pre", preprocessor),
                  ("clf", LogisticRegression(penalty=None, max_iter=10000))])

In [166]:
model.fit(X_train, y_train)

predictions = model.predict(X_test)

# Create the submission DataFrame
submission = pd.DataFrame({
    "id": test_ids,
    "label": predictions
})

# Save to CSV
submission.to_csv("submission.csv", index=False)

print("Train accuracy:", accuracy_score(y_train, model.predict(X_train)))
print(submission.head())
print("Submission shape:", submission.shape)

Train accuracy: 1.0
                                 id    label
0  0102f98c5fd7ed84754c58fe0d7f013f  Model_D
1  e2fa3e497f5bdb8c79748c7c0611fad0  Model_B
2  818a6ec782d097abc1ce67c42299d06a  Model_D
3  54e46a096cb47b5499a986477be83061  Model_B
4  5da9e7a746719fc3950c6e579c82bfa1  Model_C
Submission shape: (1600, 2)


In [167]:
assert list(X_train.columns) == list(X_test.columns)
print(submission["label"].value_counts())  

label
Model_B    414
Model_D    403
Model_C    397
Model_A    386
Name: count, dtype: int64


Each LLM writes numbers in its own style so we added features to find the patterns in their styles.

Last digit and decimal digits: each model has its own habits, like preferring round numbers or writing a certain number of decimals. A numeric column can’t capture this, so we turn the digit into a category.

Exact value as a category: each model uses its own set of typical values. Logistic regression with a numeric column only learns a straight trend, so one-hot encoding the exact value lets it learn which values are typical of which model. We group rare values to avoid overfitting.

We removed the missing-value indicators as they did not improve cross-validated accuracy, and the outlier clipping altered the exact values we now use as features.